# 02 - Silver Transform (dedupe + trim)

In [ ]:
dbutils.widgets.text("curated_base", "abfss://olistdata@olistecommdatastorage.dfs.core.windows.net/curated")
CUR = dbutils.widgets.get("curated_base").rstrip("/")

In [ ]:
# %run ./utils

In [ ]:
from pyspark.sql import functions as F

tables = ["orders","customers","order_items","order_payments","order_reviews","products","sellers"]
for t in tables:
    df = spark.read.format("delta").load(f"{CUR}/bronze/{t}")
    df = df.dropDuplicates()
    df = df.select([F.trim(F.col(c)).alias(c) if dt=="string" else F.col(c) for c,dt in df.dtypes])
    df.write.format("delta").mode("overwrite").save(f"{CUR}/silver/{t}")

# simple checks
orders_sv = spark.read.format("delta").load(f"{CUR}/silver/orders")
dup = orders_sv.groupBy("order_id").count().where("count>1").count()
assert dup == 0, f"Duplicates in silver.orders: {dup}"
print("Silver done. Sample:")
orders_sv.show(10, truncate=False)